In [1]:
import os, gc, glob, warnings
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
from collections import defaultdict
warnings.filterwarnings('ignore')

SAVE_DIR    = '/srv/work'
ARCHIVOS    = sorted(
    glob.glob(f'{SAVE_DIR}/te_*_raw_may_w*.parquet') +
    glob.glob(f'{SAVE_DIR}/te_*_raw_w[0-9].parquet')
)
METRICS_NET = ['network.latency', 'network.loss', 'network.jitter']
BD_WINDOWS  = [7, 14, 21, 28]
START_DATE  = pd.Timestamp('2026-05-01', tz='UTC')
WINDOW_MIN  = 30
SAMPLES_W   = 6
WIN_NS      = WINDOW_MIN * 60 * int(1e9)

MIN_MAD = {'network.latency': 0.005, 'network.loss': 0.5, 'network.jitter': 0.1}

GUARDRAILS_V7 = {
    'network.latency': {'media': 0.250, 'alta': 0.350},
    'network.loss':    {'media': 3.0,   'alta': 8.0},
    'network.jitter':  {'media': 7.0,   'alta': 10.0},
}

K_TRIPLETS   = [(1.0, 2.0, 3.5), (1.5, 2.5, 4.0), (2.0, 3.0, 5.0)]
K_TRIPLETS_C = [(0.5, 2.0, 3.5)] + K_TRIPLETS

PERS_BASE = 4
PERS_B    = 3

PROVIDER_MAP = [('aws', 'AWS'), ('azure', 'Azure'), ('gcp', 'GCP')]
COLS_CAL    = ['_time', 'source_agent', 'cloud_provider', 'region', 'metric', 'value']
COLS_ROUTES = ['source_agent', 'cloud_provider', 'region', 'metric']

print(f'Archivos     : {len(ARCHIVOS)}  (esperado: 39 = 30 train/val + 9 test w6/w7/w8)')
print(f'BD_WINDOWS   : {BD_WINDOWS}')
print(f'K_TRIPLETS_C : {K_TRIPLETS_C}')
print(f'GUARDRAILS v7 (corregidos):')
for m, g in GUARDRAILS_V7.items():
    print(f'  {m}: {g}')


Archivos     : 39  (esperado: 39 = 30 train/val + 9 test w6/w7/w8)
BD_WINDOWS   : [7, 14, 21, 28]
K_TRIPLETS_C : [(0.5, 2.0, 3.5), (1.0, 2.0, 3.5), (1.5, 2.5, 4.0), (2.0, 3.0, 5.0)]
GUARDRAILS v7 (corregidos):
  network.latency: {'media': 0.25, 'alta': 0.35}
  network.loss: {'media': 3.0, 'alta': 8.0}
  network.jitter: {'media': 7.0, 'alta': 10.0}


In [2]:
all_routes_full = set()

for f in ARCHIVOS:
    tbl = pq.read_table(f, columns=COLS_ROUTES, filters=[('metric', 'in', METRICS_NET)])
    df = tbl.to_pandas(); del tbl; gc.collect()
    df = df.dropna(subset=['source_agent'])
    for row in df[['source_agent','cloud_provider','region']].drop_duplicates().itertuples(index=False):
        all_routes_full.add((row.source_agent, row.cloud_provider, row.region))
    del df; gc.collect()

routes_df = pd.DataFrame(sorted(all_routes_full),
                         columns=['source_agent','cloud_provider','region'])
sampled_full_df = routes_df.copy()

print(f'Rutas totales: {len(sampled_full_df)}')
print(sampled_full_df.groupby('cloud_provider').size())


Rutas totales: 1035
cloud_provider
AWS      405
Azure    315
GCP      315
dtype: int64


In [3]:
import time

def _wins_metric(ev_t, ev_v, thr_dict, samples_w, pers):
    """Etiqueta ventanas de una métrica via np.bincount. {(k1,k2,k3): (nn,nb,nm,na)}"""
    if len(ev_t) < samples_w * 2:
        return {}
    n = int((ev_t[-1] - ev_t[0]) // WIN_NS)
    if n == 0:
        return {}
    wi = ((ev_t - ev_t[0]) // WIN_NS).astype(np.int64)
    mk = wi < n
    wi, vv = wi[mk], ev_v[mk]
    cnt = np.bincount(wi, minlength=n)
    valid = cnt >= samples_w
    if not valid.any():
        return {}
    out = {}
    for (k1, k2, k3), (tb, tm, ta) in thr_dict.items():
        ca = np.bincount(wi, weights=(vv > ta).astype(float), minlength=n)
        cm = np.bincount(wi, weights=(vv > tm).astype(float), minlength=n)
        cb = np.bincount(wi, weights=(vv > tb).astype(float), minlength=n)
        lbl = np.where(ca >= pers, 3,
              np.where(cm >= pers, 2,
              np.where(cb >= pers, 1, 0)))
        lv = lbl[valid]
        tot = len(lv)
        if tot > 0:
            out[(k1,k2,k3)] = (int((lv==0).sum()), int((lv==1).sum()),
                               int((lv==2).sum()), int((lv==3).sum()))
    return out


def _wins_or(pre_ev, thr_per_metric, k_triplets, samples_w, pers):
    """OR logic entre métricas via np.bincount. {(k1,k2,k3): (nn,nb,nm,na)}"""
    if not pre_ev:
        return {}
    all_t = np.concatenate([t for t, _ in pre_ev.values()])
    t0, t1 = all_t.min(), all_t.max()
    n = int((t1 - t0) // WIN_NS)
    if n == 0:
        return {}
    out = {}
    for (k1, k2, k3) in k_triplets:
        lbl = np.zeros(n, dtype=np.int8)
        for metric, (ev_t, ev_v) in pre_ev.items():
            d = thr_per_metric.get(metric, {})
            if (k1, k2, k3) not in d:
                continue
            tb, tm, ta = d[(k1, k2, k3)]
            wi = ((ev_t - t0) // WIN_NS).astype(np.int64)
            mk = wi < n
            wi2, vv2 = wi[mk], ev_v[mk]
            cnt = np.bincount(wi2, minlength=n)
            ca  = np.bincount(wi2, weights=(vv2 > ta).astype(float), minlength=n)
            cm  = np.bincount(wi2, weights=(vv2 > tm).astype(float), minlength=n)
            cb  = np.bincount(wi2, weights=(vv2 > tb).astype(float), minlength=n)
            ml  = np.where((cnt>=samples_w)&(ca>=pers), 3,
                  np.where((cnt>=samples_w)&(cm>=pers), 2,
                  np.where((cnt>=samples_w)&(cb>=pers), 1, 0))).astype(np.int8)
            lbl = np.maximum(lbl, ml)
        nn,nb,nm,na = (int((lbl==i).sum()) for i in range(4))
        if nn+nb+nm+na > 0:
            out[(k1,k2,k3)] = (nn, nb, nm, na)
    return out


results_A, results_B, results_C = [], [], []
t0_total = time.time()

for pkey, pname in PROVIDER_MAP:
    pfiles  = sorted([f for f in ARCHIVOS if f'/te_{pkey}_' in f])
    proutes = sampled_full_df[sampled_full_df.cloud_provider == pname].copy()
    print(f'\n{"═"*60}')
    print(f'── {pname}: {len(pfiles)} archivos, {len(proutes)} rutas ──')
    t0_prov = time.time()

    route_data = defaultdict(lambda: defaultdict(lambda: {'times': [], 'values': []}))
    for i, f in enumerate(pfiles):
        print(f'  Cargando [{i+1:02d}/{len(pfiles)}] {os.path.basename(f)}...', flush=True)
        tbl = pq.read_table(f, columns=COLS_CAL, filters=[('metric', 'in', METRICS_NET)])
        df  = tbl.to_pandas(); del tbl; gc.collect()
        df['_time'] = pd.to_datetime(df['_time'], utc=True, errors='coerce')
        df['value'] = pd.to_numeric(df['value'], errors='coerce')
        df = df.dropna(subset=['_time', 'value', 'source_agent'])
        df = df.merge(proutes, on=['source_agent','cloud_provider','region'], how='inner')
        if df.empty:
            del df; gc.collect(); continue
        for (ag, cl, rg, mt), grp in df.groupby(['source_agent','cloud_provider','region','metric']):
            route_data[(ag,cl,rg)][mt]['times'].extend(grp['_time'].tolist())
            route_data[(ag,cl,rg)][mt]['values'].extend(grp['value'].tolist())
        del df; gc.collect()
        print(f'    ✓ rutas: {len(route_data)}', flush=True)

    print(f'\n  Calibrando {len(route_data)} rutas...', flush=True)
    cnt_a = cnt_b = cnt_c = 0
    t0_cal = time.time()

    for idx, ((agent, cloud, region), mdata) in enumerate(route_data.items()):
        if idx % 100 == 0 and idx > 0:
            print(f'  Ruta {idx}/{len(route_data)} — {time.time()-t0_cal:.0f}s', flush=True)

        pre = {}
        for metric, rd in mdata.items():
            t = np.array([x.value for x in rd['times']], dtype='int64')
            v = np.array(rd['values'], dtype=float)
            if len(v) < SAMPLES_W * 2:
                continue
            o = np.argsort(t)
            pre[metric] = (t[o], v[o])
        if not pre:
            continue

        for bd in BD_WINDOWS:
            bd_ns        = bd * 24 * 3600 * int(1e9)
            baseline_end = int(START_DATE.value) + bd_ns

            thr = {}
            for metric, (t, v) in pre.items():
                base = v[t < baseline_end]
                if len(base) < 30:
                    continue
                p95 = float(np.percentile(base, 95))
                mad = max(float(np.median(np.abs(base - np.median(base)))), MIN_MAD[metric])
                thr[metric] = {
                    (k1,k2,k3): (
                        max(p95+k1*mad, GUARDRAILS_V7[metric].get('baja', 0)),
                        max(p95+k2*mad, GUARDRAILS_V7[metric]['media']),
                        max(p95+k3*mad, GUARDRAILS_V7[metric]['alta']),
                    )
                    for k1,k2,k3 in K_TRIPLETS_C
                }
            if not thr:
                continue

            pre_ev = {}
            for metric, (t, v) in pre.items():
                ev_t = t[t >= baseline_end]
                ev_v = v[t >= baseline_end]
                if len(ev_t) >= SAMPLES_W:
                    pre_ev[metric] = (ev_t, ev_v)

            for metric, (ev_t, ev_v) in pre_ev.items():
                if metric not in thr:
                    continue
                res_b = _wins_metric(ev_t, ev_v,
                    {k: thr[metric][k] for k in K_TRIPLETS if k in thr[metric]},
                    SAMPLES_W, PERS_B)
                for (k1,k2,k3),(nn,nb,nm,na) in res_b.items():
                    tot = nn+nb+nm+na
                    results_B.append(dict(
                        source_agent=agent, cloud_provider=cloud, region=region,
                        metric=metric, bd=bd, k1=k1, k2=k2, k3=k3, n_windows=tot,
                        pct_normal=round(nn/tot*100,2), pct_baja=round(nb/tot*100,2),
                        pct_media=round(nm/tot*100,2), pct_alta=round(na/tot*100,2),
                    ))
                    cnt_b += 1
                res_c = _wins_metric(ev_t, ev_v, thr[metric], SAMPLES_W, PERS_BASE)
                for (k1,k2,k3),(nn,nb,nm,na) in res_c.items():
                    tot = nn+nb+nm+na
                    results_C.append(dict(
                        source_agent=agent, cloud_provider=cloud, region=region,
                        metric=metric, bd=bd, k1=k1, k2=k2, k3=k3, n_windows=tot,
                        pct_normal=round(nn/tot*100,2), pct_baja=round(nb/tot*100,2),
                        pct_media=round(nm/tot*100,2), pct_alta=round(na/tot*100,2),
                    ))
                    cnt_c += 1

            thr_base = {m: {k: thr[m][k] for k in K_TRIPLETS if k in thr[m]}
                        for m in pre_ev if m in thr}
            res_a = _wins_or(pre_ev, thr_base, K_TRIPLETS, SAMPLES_W, PERS_BASE)
            for (k1,k2,k3),(nn,nb,nm,na) in res_a.items():
                tot = nn+nb+nm+na
                results_A.append(dict(
                    source_agent=agent, cloud_provider=cloud, region=region,
                    bd=bd, k1=k1, k2=k2, k3=k3, n_windows=tot,
                    pct_normal=round(nn/tot*100,2), pct_baja=round(nb/tot*100,2),
                    pct_media=round(nm/tot*100,2), pct_alta=round(na/tot*100,2),
                ))
                cnt_a += 1

    del route_data; gc.collect()
    print(f'\n✅ {pname}: A={cnt_a}  B={cnt_b}  C={cnt_c}  ({time.time()-t0_prov:.0f}s)', flush=True)

df_A = pd.DataFrame(results_A)
df_B = pd.DataFrame(results_B)
df_C = pd.DataFrame(results_C)
ela = time.time() - t0_total
print(f'\n{"═"*60}')
print(f'Registros — A:{len(df_A)}  B:{len(df_B)}  C:{len(df_C)}')
print(f'Tiempo total: {ela:.0f}s ({ela/60:.1f} min)')
print(f'{"═"*60}')



════════════════════════════════════════════════════════════
── AWS: 13 archivos, 405 rutas ──
  Cargando [01/13] te_aws_raw_may_w1.parquet...
    ✓ rutas: 405
  Cargando [02/13] te_aws_raw_may_w2.parquet...
    ✓ rutas: 405
  Cargando [03/13] te_aws_raw_may_w3.parquet...
    ✓ rutas: 405
  Cargando [04/13] te_aws_raw_may_w4.parquet...
    ✓ rutas: 405
  Cargando [05/13] te_aws_raw_may_w5.parquet...
    ✓ rutas: 405
  Cargando [06/13] te_aws_raw_w1.parquet...
    ✓ rutas: 405
  Cargando [07/13] te_aws_raw_w2.parquet...
    ✓ rutas: 405
  Cargando [08/13] te_aws_raw_w3.parquet...
    ✓ rutas: 405
  Cargando [09/13] te_aws_raw_w4.parquet...
    ✓ rutas: 405
  Cargando [10/13] te_aws_raw_w5.parquet...
    ✓ rutas: 405
  Cargando [11/13] te_aws_raw_w6.parquet...
    ✓ rutas: 405
  Cargando [12/13] te_aws_raw_w7.parquet...
    ✓ rutas: 405
  Cargando [13/13] te_aws_raw_w8.parquet...
    ✓ rutas: 405

  Calibrando 405 rutas...
  Ruta 100/405 — 17s
  Ruta 200/405 — 35s
  Ruta 300/405 — 52s
 

In [4]:

def show(df, label):
    cols = ['pct_normal','pct_baja','pct_media','pct_alta']
    s = df.groupby(['bd','k1','k2','k3'])[cols].mean().round(2).reset_index()
    s['no_normal'] = (100 - s['pct_normal']).round(2)
    s = s.sort_values('no_normal', ascending=False)
    print(f'\n{"═"*72}\n{label}\n{"═"*72}')
    for _, r in s.iterrows():
        ok = r.pct_normal>=80 and 5<=r.no_normal<=15 and r.pct_alta>0
        print(f'{"✅" if ok else "  "}  bd={int(r.bd):2d}  k=({r.k1},{r.k2},{r.k3})'
              f'  Normal={r.pct_normal:.1f}%  Baja={r.pct_baja:.1f}%'
              f'  Media={r.pct_media:.1f}%  Alta={r.pct_alta:.1f}%'
              f'  No-Normal={r.no_normal:.1f}%')

print('REFERENCIA — v6 Opcion A OR logic (guardrail latencia INACTIVO 250s):')
print('  bd= 7  k=(1.0,2.0,3.5)  Normal=88.6%  No-Normal=11.4%')
print('  bd=14  k=(1.0,2.0,3.5)  Normal=90.1%  No-Normal= 9.9%')
print('  bd=21  k=(1.0,2.0,3.5)  Normal=89.8%  No-Normal=10.2%')

show(df_A, 'OPCION A — OR logic (max metrica por ventana) | >=4/6 | v7 guardrail corregido')
show(df_B, 'OPCION B — per-metrica | persistencia >=3/6 | v7 guardrail corregido')
show(df_C[df_C.k1==0.5], 'OPCION C — k1=0.5 (umbral baja sensible) | >=4/6')
show(df_C[df_C.k1!=0.5], 'OPCION C — triplets base | >=4/6')


REFERENCIA — v6 Opcion A OR logic (guardrail latencia INACTIVO 250s):
  bd= 7  k=(1.0,2.0,3.5)  Normal=88.6%  No-Normal=11.4%
  bd=14  k=(1.0,2.0,3.5)  Normal=90.1%  No-Normal= 9.9%
  bd=21  k=(1.0,2.0,3.5)  Normal=89.8%  No-Normal=10.2%

════════════════════════════════════════════════════════════════════════
OPCION A — OR logic (max metrica por ventana) | >=4/6 | v7 guardrail corregido
════════════════════════════════════════════════════════════════════════
✅  bd= 7  k=(1.0,2.0,3.5)  Normal=88.0%  Baja=8.0%  Media=1.9%  Alta=2.1%  No-Normal=12.0%
✅  bd=21  k=(1.0,2.0,3.5)  Normal=89.3%  Baja=6.9%  Media=1.8%  Alta=2.0%  No-Normal=10.7%
✅  bd=14  k=(1.0,2.0,3.5)  Normal=89.4%  Baja=6.8%  Media=1.7%  Alta=2.0%  No-Normal=10.6%
✅  bd= 7  k=(1.5,2.5,4.0)  Normal=89.7%  Baja=6.4%  Media=2.0%  Alta=2.0%  No-Normal=10.3%
✅  bd=28  k=(1.0,2.0,3.5)  Normal=89.7%  Baja=6.7%  Media=1.7%  Alta=1.9%  No-Normal=10.3%
✅  bd=21  k=(1.5,2.5,4.0)  Normal=90.9%  Baja=5.5%  Media=1.8%  Alta=1.9%  No-Nor

In [5]:

BD_FINAL_V7        = 7
K1_FINAL_V7        = 1.0
K2_FINAL_V7        = 2.0
K3_FINAL_V7        = 3.5
MIN_PERSISTENCE_V7 = 4
LABEL_LOGIC_V7     = 'OR'

GUARDRAILS_V7_FINAL = {
    'network.latency': {'media': 0.250, 'alta': 0.350},
    'network.loss':    {'media': 3.0,   'alta': 8.0},
    'network.jitter':  {'media': 7.0,   'alta': 10.0},
}
MIN_MAD_V7 = {
    'network.latency': 0.005,
    'network.loss':    0.5,
    'network.jitter':  0.1,
}

print('=== PARAMETROS FINALES v7 ===')
print(f'BD_FINAL        : {BD_FINAL_V7} dias')
print(f'K (baja/med/alt): ({K1_FINAL_V7}, {K2_FINAL_V7}, {K3_FINAL_V7})')
print(f'MIN_PERSISTENCE : {MIN_PERSISTENCE_V7}/6 ventanas')
print(f'LABEL_LOGIC     : {LABEL_LOGIC_V7}')
print(f'GUARDRAILS      : {GUARDRAILS_V7_FINAL}')
print()
print('Copiar estos valores al inicio de: 03 etiquetado v3.ipynb')
print()

print('=== COMPARACION v6 vs v7 ===')
print(f'  BD       : v6=7d           -> v7={BD_FINAL_V7}d')
print(f'  K        : v6=(1.0,2.0,3.5)-> v7=({K1_FINAL_V7},{K2_FINAL_V7},{K3_FINAL_V7})')
print(f'  Guardrail latencia: v6=250s (INACTIVO) -> v7=0.250s (ACTIVO)')


=== PARAMETROS FINALES v7 ===
BD_FINAL        : 7 dias
K (baja/med/alt): (1.0, 2.0, 3.5)
MIN_PERSISTENCE : 4/6 ventanas
LABEL_LOGIC     : OR
GUARDRAILS      : {'network.latency': {'media': 0.25, 'alta': 0.35}, 'network.loss': {'media': 3.0, 'alta': 8.0}, 'network.jitter': {'media': 7.0, 'alta': 10.0}}

Copiar estos valores al inicio de: 03 etiquetado v3.ipynb

=== COMPARACION v6 vs v7 ===
  BD       : v6=7d           -> v7=7d
  K        : v6=(1.0,2.0,3.5)-> v7=(1.0,2.0,3.5)
  Guardrail latencia: v6=250s (INACTIVO) -> v7=0.250s (ACTIVO)
